In [ ]:
# ── Cell 1 ───────────────────────────────────────────────────────────
import os, sys
OUT_DIR = "ps5_validator_reliability_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    def __init__(self, j, f): self._j, self._f, self._is_tee = j, f, True
    def write(self, d):
        try: self._j.write(d); self._j.flush()
        except: pass
        try: self._f.write(d); self._f.flush()
        except: pass
    def flush(self):
        try: self._j.flush()
        except: pass
    def __getattr__(self, n): return getattr(self._j, n)

if not getattr(sys.stdout, "_is_tee", False):
    _lf = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _lf)
    sys.stderr = _Tee(sys.stderr, _lf)
    print(f"[capture] -> {OUT_DIR}/console.log")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig = _plt.show
        def _show(*a, _c=[0], **k):
            for n in _plt.get_fignums():
                fig = _plt.figure(n)
                t = ((fig._suptitle.get_text().strip() if fig._suptitle else None)
                     or (fig.axes[0].get_title().strip() if fig.axes else None)
                     or f"fig_{_c[0]:02d}")
                s = "".join(c if c.isalnum() or c in "-_ " else "_" for c in t).strip().replace(" ","_")[:60]
                p = os.path.join(OUT_DIR, f"{s}_{_c[0]:02d}.png")
                try: fig.savefig(p, dpi=110, bbox_inches="tight"); print(f"[fig] {p}"); _c[0]+=1
                except Exception as e: print(f"[fig err] {e}")
            return _orig(*a, **k)
        _show._wrapped = True; _plt.show = _show
    print(f"[capture] matplotlib -> {OUT_DIR}/")
except Exception as e: print("[capture] matplotlib not ready:", e)


# PS5 — VALIDATOR Device Reliability Analysis
Survival analysis, fault risk ranking, reliability growth, and RUL for **Ticket Vending Machines**.

**Data sources**:
- `silver.device_incident_features_daily` (S24) — event-level incidents with fault codes (`event_code_id`)
- `gold.device_ps1_daily` — device-type mapping (`mars_device_category`); also used as TBF fallback via `days_since_last_incident`

**Sections**: KM Survival · Weibull · Cox PH · Risk Ranking · Severity · Fault Heatmap · Crow-AMSAA · RBD · RUL

In [ ]:
# ── Cell 2 ───────────────────────────────────────────────────────────
# ── Install ───────────────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "lifelines", "scikit-learn", "matplotlib>=3.9", "seaborn",
    "pandas", "numpy", "pyarrow", "boto3", "s3fs", "scipy",
    "networkx", "mlxtend",
    "mlflow>=2.13", "sagemaker-mlflow",
    "sagemaker>=2.220,<3.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("Install complete.")

In [ ]:
# ── Cell 3 ───────────────────────────────────────────────────────────
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx

from lifelines import KaplanMeierFitter, WeibullFitter, NelsonAalenFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sklearn.preprocessing import StandardScaler
from scipy.stats import weibull_min

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_style("whitegrid")
print("Imports OK")

from sagemaker.session                        import Session
from sagemaker                                 import get_execution_role
from sagemaker.feature_store.feature_group    import FeatureGroup
from sagemaker.feature_store.feature_definition import (
    FeatureDefinition, FeatureTypeEnum,
)

In [ ]:
# ── Cell 4 ───────────────────────────────────────────────────────────
# ── Configuration ─────────────────────────────────────────────────────────────
DEVICE_TYPE  = "VALIDATOR"
REGION       = "us-east-1"

# device_event_enriched: raw hardware/software fault events logged by the device.
# Preferred over ServiceNow incident_history (delayed, incomplete).
SILVER_EVT_S3 = "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched"
GOLD_PS1_S3   = "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/gold/device_ps1_daily"

S3_OPTS      = {"client_kwargs": {"region_name": REGION}}
DEV_COL      = "mars_device_category"
DEV_KEY      = "DEVICE_KEY"
DEV_ID_COL   = "DEVICE_ID"
FC_COL       = "EVENT_TYPE_NAME"  # Cubic official name; always populated (numeric EVENT_TYPE_ID is often null)
DATE_COL     = "event_date"
TOP_N        = 10
RANDOM_STATE = 42

print(f"Device type : {DEVICE_TYPE}")
print(f"Silver path : {SILVER_EVT_S3}")
print(f"Gold path   : {GOLD_PS1_S3}")

# ── MLflow setup ──────────────────────────────────────────────────────────────
import mlflow, pickle
from mlflow.tracking import MlflowClient
import boto3 as _boto3

MLFLOW_SERVER_ARN = (
    "arn:aws:sagemaker:us-east-1:170202974600:"
    "mlflow-tracking-server/cubic-mars-mlflow-server-dev"
)
try:
    _boto3.client("sagemaker", region_name=REGION).describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )
    print("MLflow managed server: available")
except Exception:
    print("MLflow managed server not found — local EFS fallback will be used")

mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)
EXPERIMENT_NAME   = "chicago-ps5-validator-rul"
MLFLOW_MODEL_NAME = "ps5-validator-rul"
COMMON_TAGS = {
    "ps": "PS5", "device_type": DEVICE_TYPE,
    "city": "chicago", "project": "CUBIC MARS AWS v6",
}
mlflow.set_experiment(EXPERIMENT_NAME)
cph = None; CHAMPION_RUN_ID = None; CHAMPION_CI = None
print(f"MLflow experiment : {EXPERIMENT_NAME}")
print(f"MLflow model      : {MLFLOW_MODEL_NAME}")
# ── SageMaker Feature Store config ──────────────────────────────────────
SM_SESSION    = Session()
ROLE          = get_execution_role()
ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
FS_GROUP_NAME   = "chicago-ps5-validator-rul-features"
print(f"Feature Store group : {FS_GROUP_NAME}")
# ── Routine / maintenance events to exclude from fault analysis ──────────────
EXCLUDE_EVENTS = {'CoinTubeDumped', 'In Service', 'Alarm Keyed Off', 'Table Switched', 'Mnt Door Open', 'CoinVault Op/Cl', 'ToppedOff', 'Emp Logon', 'CommandedReboot', 'Time Sync', 'CBX Rmvd/Insrtd', 'Tube Rmv/Insrtd', 'BBX Rmvd/Insrtd'}
print(f"Exclusion list  : {len(EXCLUDE_EVENTS)} routine event types")


In [ ]:
# ── Cell 5 ───────────────────────────────────────────────────────────
# ── Load Data ─────────────────────────────────────────────────────────────────
USING_GOLD_FALLBACK = False
TBF_COL_GOLD = "days_since_last_incident"

# Candidate paths — device_event_enriched first (raw device telemetry, NOT ServiceNow tickets).
# incident_history = ServiceNow tickets only; misses many device failures and has coarser timing.
_SILVER_CANDIDATES = [
    "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched",
    "s3://cubic-mars-pm-s3-datalake-dev-silver-170202974600/chicago/silver/device_event_enriched",
    # incident_history as last-resort silver fallback (ServiceNow tickets)
    "s3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/incident_history",
]

# Reject tables that have no real event/fault code (only aggregate count columns)
_COUNT_ONLY_COLS = {"distinct_event_codes_30d"}

# Columns to read from silver — prunes S3 I/O; unknown extras auto-excluded via schema probe
_WANT_COLS = list(set(filter(None, [
    DEV_KEY, DEV_ID_COL, DEV_COL, FC_COL, DATE_COL,
    "EVENT_DTM", "event_date", "event_datetime", "event_code_id", "EVENT_TYPE_ID",
    "mars_device_category", "DEVICE_ID",
    "is_chargeable", "priority", "is_major_incident",
])))

def _has_real_fault_code(cols):
    for c in cols:
        cl = c.lower()
        if ("code" in cl or "fault" in cl or "event_type" in cl) and c not in _COUNT_ONLY_COLS:
            return True
    return False

# Pre-load gold for dev_map — lets us filter silver at read time instead of post-load.
# Gold PS1 is device-level daily aggregates (small); silver is event-level (large).
print("Pre-loading device-type mapping from gold ...")
_gold_pre = pd.read_parquet(GOLD_PS1_S3, storage_options=S3_OPTS)
_map_cols_pre = [c for c in [DEV_KEY, DEV_ID_COL, DEV_COL, "DEVICE_NAME"] if c in _gold_pre.columns]
dev_map = _gold_pre[_map_cols_pre].drop_duplicates(subset=[DEV_KEY]).copy()
if DEV_COL in dev_map.columns:
    dev_map[DEV_COL] = dev_map[DEV_COL].astype(str).str.strip().str.upper()
    _dev_keys = dev_map[dev_map[DEV_COL] == DEVICE_TYPE][DEV_KEY].unique().tolist()
else:
    _dev_keys = None
del _gold_pre
print(f"  Pre-filter: {len(_dev_keys) if _dev_keys else '(all)'} {DEVICE_TYPE} device keys")

df_inc = None
SILVER_EVT_S3_USED = None
for _path in _SILVER_CANDIDATES:
    try:
        print(f"Trying: {_path} ...")
        _pq_filter = [(DEV_KEY, "in", _dev_keys)] if _dev_keys else None
        try:
            import pyarrow.parquet as _pq_mod
            import s3fs as _s3fs_mod
            _fs_probe = _s3fs_mod.S3FileSystem(client_kwargs={"region_name": REGION})
            _pq_names = set(
                _pq_mod.ParquetDataset(
                    _path.replace("s3://", ""), filesystem=_fs_probe
                ).schema.names
            )
            # Also keep any date/time-like column so auto-detection always finds one
            _DATE_KW = ("date", "time", "_at", "_on", "_dtm", "_dt", "stamp", "created", "opened", "closed", "resolved")
            _date_extras = {c for c in _pq_names if any(k in c.lower() for k in _DATE_KW)}
            _cols_read = list((set(_WANT_COLS) & _pq_names) | _date_extras) or None
            if _cols_read:
                print(f"  Column pruning: {len(_cols_read)}/{len(_pq_names)} cols")
            _tmp = pd.read_parquet(_path, storage_options=S3_OPTS,
                                   filters=_pq_filter, columns=_cols_read)
        except Exception:
            try:
                _tmp = pd.read_parquet(_path, storage_options=S3_OPTS, filters=_pq_filter)
            except Exception:
                _tmp = pd.read_parquet(_path, storage_options=S3_OPTS)
        print(f"  OK: {_tmp.shape[0]:,} rows x {_tmp.shape[1]} cols")
        print(f"  Columns: {_tmp.columns.tolist()}")
        if not _has_real_fault_code(_tmp.columns):
            print(f"  SKIP: no real fault/event code column — trying next path")
            continue
        df_inc = _tmp
        SILVER_EVT_S3_USED = _path
        break
    except Exception as _e:
        print(f"  FAIL: {str(_e)[:120]}")

if df_inc is not None:
    # Silver loaded — dev_map already built in pre-load above
    print(f"\nLoaded: {SILVER_EVT_S3_USED}")

    # Normalise DEVICE_KEY column name
    if DEV_KEY not in df_inc.columns:
        _dk_cands = [c for c in df_inc.columns if c.lower() == DEV_KEY.lower()]
        if _dk_cands:
            df_inc = df_inc.rename(columns={_dk_cands[0]: DEV_KEY})

    # Auto-detect date column (device_event_enriched uses event_date or event_datetime)
    if DATE_COL not in df_inc.columns:
        # Exact name candidates — uppercase variants first (silver uses uppercase columns)
        for _dc in ["EVENT_DTM", "EVENT_DATE", "EVENT_DATETIME", "event_date", "event_datetime",
                    "incident_date", "INCIDENT_DATE", "transit_day", "created_at", "date"]:
            if _dc in df_inc.columns:
                DATE_COL = _dc
                break
        # Broader fallback — exclude ETL/pipeline metadata columns (EDW_*, *_UPDATED_*, *_LOADED_*)
        if DATE_COL not in df_inc.columns:
            _date_kw  = ("date", "time", "stamp", "_at", "_dtm", "_dt", "created", "opened", "closed")
            _etl_excl = ("edw", "updated", "loaded", "inserted", "etl", "modified",
                         "ingested", "processed", "refreshed")
            DATE_COL = next(
                (c for c in df_inc.columns
                 if any(k in c.lower() for k in _date_kw)
                 and not any(e in c.lower() for e in _etl_excl)),
                DATE_COL
            )
        if DATE_COL in df_inc.columns:
            print(f"  Date column auto-detected: {DATE_COL}")
        else:
            print(f"  [WARN] No event date column found. Available: {df_inc.columns.tolist()}")

    if DATE_COL in df_inc.columns:
        df_inc[DATE_COL] = pd.to_datetime(df_inc[DATE_COL], errors="coerce")
    else:
        print(f"  [ERROR] DATE_COL '{DATE_COL}' not in silver — TBF cannot be computed")
    df = df_inc.merge(dev_map, on=DEV_KEY, how="left")
    # Resolve suffix conflict: both silver and dev_map have mars_device_category
    # pandas renames them _x (silver) and _y (gold) — use silver as truth for events
    _sc = DEV_COL + "_x"  # silver's category
    _gc = DEV_COL + "_y"  # gold's  category
    if _sc in df.columns:
        df[DEV_COL] = df[_sc].astype(str).str.strip().str.upper()
        df = df.drop(columns=[c for c in [_sc, _gc] if c in df.columns])
    elif _gc in df.columns:
        df[DEV_COL] = df[_gc].astype(str).str.strip().str.upper()
        df = df.drop(columns=[_gc])
    if DEV_COL in df.columns:
        df = df[df[DEV_COL] == DEVICE_TYPE].copy()
        print(f"  Filtered to {DEVICE_TYPE}: {len(df):,} rows")

    # Confirm fault/event code column
    if FC_COL not in df.columns:
        _cands = [c for c in df.columns
                  if ("code" in c.lower() or "fault" in c.lower() or "event_type" in c.lower())
                  and c not in _COUNT_ONLY_COLS]
        FC_COL = _cands[0] if _cands else None
        if FC_COL:
            print(f"  Event code column: {FC_COL} (auto-detected)")
    else:
        print(f"  Event code column: {FC_COL}")
    if FC_COL:
        df[FC_COL] = df[FC_COL].astype(str)

    print(f"\n{DEVICE_TYPE} device events (silver): {len(df):,} rows")
    print(f"Unique event codes : {df[FC_COL].nunique():,}" if FC_COL else "[WARN] No event code column found")
    print(f"Unique devices     : {df[DEV_KEY].nunique():,}")
    if DATE_COL in df.columns and df[DATE_COL].notna().any():
        print(f"Date range         : {df[DATE_COL].min().date()} -> {df[DATE_COL].max().date()}")

else:
    # ── GOLD FALLBACK ─────────────────────────────────────────────────────────
    print("\n[GOLD FALLBACK] Silver device_event_enriched not accessible.")
    print("  Using gold PS1 daily — has real event_code_id + days_since_last_incident")
    USING_GOLD_FALLBACK = True

    df_gold_all = pd.read_parquet(GOLD_PS1_S3, storage_options=S3_OPTS)
    print(f"  Gold PS1: {df_gold_all.shape[0]:,} rows x {df_gold_all.shape[1]} cols")

    for _date_cand in ["transit_day", "event_date", "date", "incident_date"]:
        if _date_cand in df_gold_all.columns:
            DATE_COL = _date_cand
            break
    print(f"  Date column: {DATE_COL}")

    if DEV_COL in df_gold_all.columns:
        df_gold_all[DEV_COL] = df_gold_all[DEV_COL].astype(str).str.strip().str.upper()
        df = df_gold_all[df_gold_all[DEV_COL] == DEVICE_TYPE].copy()
    else:
        df = df_gold_all.copy()

    map_cols = [c for c in [DEV_KEY, DEV_ID_COL, DEV_COL] if c in df_gold_all.columns]
    dev_map = df_gold_all[map_cols].drop_duplicates(subset=[DEV_KEY]).copy()

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    del df_gold_all

    if TBF_COL_GOLD not in df.columns:
        _tbf_cands = [c for c in df.columns if "since" in c.lower() or "tbf" in c.lower()
                      or ("days" in c.lower() and "incident" in c.lower())]
        TBF_COL_GOLD = _tbf_cands[0] if _tbf_cands else None
        print(f"  [WARN] TBF column not found. Candidates: {_tbf_cands}")

    # Gold PS1 has no EVENT_TYPE_NAME — reset FC_COL to a gold-compatible column
    FC_COL = None
    for _fc_cand in ["EVENT_TYPE_NAME", "event_code_id", "fault_code", "event_type"]:
        if _fc_cand in df.columns:
            FC_COL = _fc_cand
            df[FC_COL] = df[FC_COL].astype(str)
            break
    if FC_COL is None:
        print("  [WARN] No event code column found in gold PS1")

    print(f"\n{DEVICE_TYPE} rows (gold fallback): {len(df):,}")
    print(f"TBF column    : {TBF_COL_GOLD}")
    print(f"Event code    : {FC_COL}")
    print(f"Unique devices: {df[DEV_KEY].nunique():,}")
    if DATE_COL in df.columns and df[DATE_COL].notna().any():
        print(f"Date range    : {df[DATE_COL].min().date()} -> {df[DATE_COL].max().date()}")
    print("\n[NOTE] To use device telemetry events, export from Databricks:"
          "\n  spark.table('mars_dev.silver.device_event_enriched')"
          "\n    .write.mode('overwrite').parquet("
          "\n      's3://cubic-mars-pm-s3-datalake-dev-gold-170202974600/chicago/silver/device_event_enriched')"
          "\n  Then re-run from Cell 4.")
# ── Exclude routine / maintenance events (not fault signals) ─────────────────
if FC_COL and FC_COL in df.columns and not USING_GOLD_FALLBACK:
    _before = len(df)
    df = df[~df[FC_COL].isin(EXCLUDE_EVENTS)].copy()
    _removed = _before - len(df)
    print()
    print(f"Exclusion filter: removed {_removed:,} routine events "
          f"({len(EXCLUDE_EVENTS)} types) → {len(df):,} fault events remain")
    if _removed == 0:
        print(f"  [INFO] No excluded events matched — verify EXCLUDE_EVENTS names")
        print(f"  Available FC_COL values (sample): {df[FC_COL].unique()[:10].tolist()}")

# ── Memory optimisation: category dtype cuts string RAM ~40% before sort ────
if FC_COL and FC_COL in df.columns:
    df[FC_COL] = df[FC_COL].astype('category')
if df[DEV_KEY].dtype == object:
    df[DEV_KEY] = df[DEV_KEY].astype('category')
print(f"Memory after category cast: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")


In [ ]:
import gc
df_inc = None; gc.collect()   # free raw silver before the sort copies

if 'df' not in dir():
    raise RuntimeError("'df' undefined — Cell 5 did not complete. Re-run Cell 5 first.")
# ── Cell 6 ───────────────────────────────────────────────────────────
# ── Compute TBF (Time Between Failures) per Device+FaultCode ─────────────────
if USING_GOLD_FALLBACK:
    # days_since_last_incident IS the TBF for rows where an incident occurred
    if TBF_COL_GOLD and TBF_COL_GOLD in df.columns:
        df_tbf = df[(df[TBF_COL_GOLD].notna()) & (df[TBF_COL_GOLD] > 0)].copy()
        df_tbf["TBF_days"] = df_tbf[TBF_COL_GOLD].astype(float)
        df_tbf["TBF_min"]  = df_tbf["TBF_days"] * 1440.0
        df_tbf["event"]    = 1
        df_sorted = df_tbf.copy()
        print(f"TBF (gold: days_since_last_incident): {len(df_tbf):,} rows")
    else:
        # No pre-computed TBF column — compute by date-diff per device
        df_sorted = df.sort_values([DEV_KEY, DATE_COL]).copy()
        df_sorted["_prev_date"] = df_sorted.groupby(DEV_KEY)[DATE_COL].shift(1)
        df_sorted["TBF_days"]   = (df_sorted[DATE_COL] - df_sorted["_prev_date"]).dt.days.astype(float)
        df_sorted["TBF_min"]    = df_sorted["TBF_days"] * 1440.0
        df_sorted["event"]      = 1
        df_tbf = df_sorted.dropna(subset=["TBF_days"]).copy()
        df_tbf = df_tbf[df_tbf["TBF_days"] > 0].copy()
        print(f"TBF (gold date-diff): {len(df_tbf):,} rows")
else:
    # Silver path — sort in-place: avoids duplicating 100M+ rows in RAM
    _group = [DEV_KEY, FC_COL] if FC_COL else [DEV_KEY]
    df.sort_values(_group + [DATE_COL], inplace=True)
    df["_prev_date"] = df.groupby(_group)[DATE_COL].shift(1)
    df["TBF_min"]    = (df[DATE_COL] - df["_prev_date"]).dt.total_seconds() / 60.0
    df["TBF_days"]   = df["TBF_min"] / 1440.0
    df["event"]      = 1
    df_tbf = df.dropna(subset=["TBF_min"])
    df_tbf = df_tbf[df_tbf["TBF_min"] > 0].copy()
    df_sorted = df_tbf     # alias for compatibility
    print(f"TBF rows: {len(df_tbf):,}")

if len(df_tbf) > 0:
    print(df_tbf["TBF_days"].describe().round(2).to_string())

# ── Print top 10 fault codes and devices with failures ────────────────────────
top10_fc   = df_tbf[FC_COL].value_counts().head(10) if (FC_COL and FC_COL in df_tbf.columns) else pd.Series(dtype=int)
top10_devs = df_tbf[DEV_KEY].value_counts().head(10)

print(f"\n{'='*55}")
print(f"TOP 10 FAULT CODES -- {DEVICE_TYPE}")
print(f"{'='*55}")
# Secondary reference: show numeric EVENT_TYPE_ID alongside the name-based FC_COL
# (avoid auto-detecting DEVICE_NAME which looks similar to EVENT_TYPE_NAME)
_evt_name_col = next(
    (c for c in ["EVENT_TYPE_ID", "event_code_id", "event_type_id"]
     if c in df_tbf.columns and c != FC_COL),
    None
)

if len(top10_fc) > 0:
    _fc_df = top10_fc.reset_index()
    _fc_df.columns = [FC_COL, 'failure_count']
    _fc_df = _fc_df.rename(columns={FC_COL: "event_type_name"})
    if _evt_name_col:
        _fc_id_map = (df_tbf[[FC_COL, _evt_name_col]].dropna(subset=[_evt_name_col])
                         .drop_duplicates(FC_COL).set_index(FC_COL)[_evt_name_col])
        _fc_df["event_code_id"] = _fc_df["event_type_name"].map(_fc_id_map)
        _fc_df = _fc_df[["event_code_id", "event_type_name", "failure_count"]]
    print(_fc_df.to_string(index=False))
else:
    print('No event code data available.')
print(f"\n{'='*55}")
print(f"TOP 10 DEVICES BY FAILURE COUNT -- {DEVICE_TYPE}")
print(f"{'='*55}")
# Device ID lookup via dev_map
_dev_id_col = DEV_ID_COL if DEV_ID_COL in dev_map.columns else None
_id_map = (dev_map[[DEV_KEY, _dev_id_col]].drop_duplicates(DEV_KEY)
           .set_index(DEV_KEY)[_dev_id_col]) if _dev_id_col else None
_dev_df = top10_devs.reset_index()
_dev_df.columns = [DEV_KEY, 'failure_count']
if _id_map is not None:
    _dev_df.insert(0, _dev_id_col, _dev_df[DEV_KEY].map(_id_map))
print(_dev_df.to_string(index=False))

TOP10_FC = top10_fc.index.tolist() if len(top10_fc) > 0 else []
SELECTED_DEVICE = top10_devs.index[0] if len(top10_devs) > 0 else None
_sel_id = (_id_map.get(SELECTED_DEVICE, "unknown")
           if (_id_map is not None and SELECTED_DEVICE is not None) else None)
print(f"\nSelected device for per-device analysis: {SELECTED_DEVICE}"
      + (f"  ({_dev_id_col}: {_sel_id})" if _sel_id else ""))

In [ ]:
# ── Cell 6c — Repeat Failure / Chronic Repeater Analysis ─────────────────────
# A repeat failure = same OOS event type on same device recurring within REPEAT_WINDOW days.
# High repeat rate = repair did not hold -> wrong root cause fix or inadequate PM.
REPEAT_WINDOW = 30  # days

if len(df_tbf) > 0 and FC_COL:
    df_repeat = df_tbf[df_tbf["TBF_days"] <= REPEAT_WINDOW].copy()

    fc_counts  = df_tbf[FC_COL].value_counts()
    fc_repeats = df_repeat[FC_COL].value_counts()
    repeat_rate = (fc_repeats / fc_counts * 100).dropna().sort_values(ascending=False)

    dev_repeat_count = df_repeat.groupby(DEV_KEY).size().sort_values(ascending=False)
    dev_total        = df_tbf.groupby(DEV_KEY).size()
    dev_repeat       = df_repeat.groupby(DEV_KEY).size()

    print("=" * 60)
    print(f"REPEAT FAILURE ANALYSIS - {DEVICE_TYPE}  (window: {REPEAT_WINDOW}d)")
    print("=" * 60)
    print(f"Total TBF intervals : {len(df_tbf):,}")
    print(f"Repeat failures     : {len(df_repeat):,}  ({len(df_repeat)/len(df_tbf)*100:.1f}%)")
    print(f"\nTop 10 Fault Codes by Repeat Rate (% recur within {REPEAT_WINDOW}d):")
    print(repeat_rate.head(10).round(1).to_string())
    print("\nTop 10 Chronic Repeater Devices:")
    print(dev_repeat_count.head(10).to_string())

    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(
        f"{DEVICE_TYPE} - Repeat Failure / Chronic Repeater Analysis (window={REPEAT_WINDOW}d)",
        fontsize=13, fontweight="bold"
    )

    _top_rr = repeat_rate.head(10)
    axes[0].barh(range(len(_top_rr)), _top_rr.values, color="#E53935", alpha=0.85)
    axes[0].set_yticks(range(len(_top_rr)))
    axes[0].set_yticklabels([f"FC={fc}" for fc in _top_rr.index], fontsize=9)
    axes[0].axvline(repeat_rate.mean(), color="black", ls="--", lw=1,
                    label=f"Fleet avg={repeat_rate.mean():.1f}%")
    axes[0].set(title=f"Repeat Rate per Fault Code (recur <={REPEAT_WINDOW}d)", xlabel="Repeat Rate (%)")
    axes[0].legend(fontsize=8)

    _top_dev_r = dev_repeat_count.head(10)
    axes[1].barh(range(len(_top_dev_r)), _top_dev_r.values, color="#FF6F00", alpha=0.85)
    axes[1].set_yticks(range(len(_top_dev_r)))
    axes[1].set_yticklabels([str(d) for d in _top_dev_r.index], fontsize=9)
    axes[1].set(title="Top 10 Chronic Repeater Devices", xlabel="Repeat Failure Count")

    _tbf_clip = df_tbf["TBF_days"].clip(upper=df_tbf["TBF_days"].quantile(0.99))
    axes[2].hist(_tbf_clip, bins=60, color="#1565C0", alpha=0.6, edgecolor="white", label="All TBF")
    axes[2].hist(df_repeat["TBF_days"].clip(upper=REPEAT_WINDOW), bins=30,
                 color="#E53935", alpha=0.8, edgecolor="white", label=f"Repeats (<={REPEAT_WINDOW}d)")
    axes[2].axvline(REPEAT_WINDOW, color="black", ls="--", lw=1.5, label=f"{REPEAT_WINDOW}d threshold")
    axes[2].set(title="TBF Distribution - Repeat vs Normal", xlabel="TBF (days)", ylabel="Count")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    repeat_summary = pd.DataFrame({
        "total_failures"  : fc_counts,
        "repeat_failures" : fc_repeats.reindex(fc_counts.index, fill_value=0),
        "repeat_rate_pct" : repeat_rate.reindex(fc_counts.index, fill_value=0).round(1),
    }).sort_values("repeat_failures", ascending=False).head(15)
    print("\nRepeat Failure Summary - Top 15 by Repeat Count:")
    print(repeat_summary.to_string())
else:
    print("[SKIP] Repeat failure analysis - no TBF or fault code data.")


In [ ]:
# ── Cell 6e – MTBF Aging Trend (Degradation Detection) ───────────────────────────
# Rolling quarterly MTBF per device. A declining slope = device is degrading.
# Negative slope -> MTBF shrinking over time -> nearing end of life.
from scipy.stats import linregress as _lr
from matplotlib.patches import Patch as _P2

TOP_N_AGING = 8

if len(df_tbf) > 0 and DATE_COL in df_tbf.columns:
    df_age = df_tbf[[DEV_KEY, DATE_COL, "TBF_days"]].copy()
    df_age[DATE_COL] = pd.to_datetime(df_age[DATE_COL])
    # pandas >= 2.2 uses QE; older uses Q
    for _qalias in ["QE", "Q"]:
        try:
            df_age["period"] = df_age[DATE_COL].dt.to_period(_qalias)
            break
        except Exception:
            continue

    period_mtbf = (
        df_age.groupby([DEV_KEY, "period"])["TBF_days"]
        .mean().reset_index()
    )
    period_mtbf["period_dt"] = period_mtbf["period"].dt.to_timestamp()

    dev_periods  = period_mtbf.groupby(DEV_KEY)["period"].count()
    top_devs_age = dev_periods[dev_periods >= 3].nlargest(TOP_N_AGING).index.tolist()

    # Vectorized slope computation via groupby.apply
    def _slope(grp):
        x = (grp["period_dt"] - grp["period_dt"].min()).dt.days.values.astype(float)
        y = grp["TBF_days"].values.astype(float)
        if len(x) >= 2 and x.std() > 0:
            return _lr(x, y).slope
        return float("nan")

    _valid = dev_periods[dev_periods >= 3].index
    slopes_s = (period_mtbf[period_mtbf[DEV_KEY].isin(_valid)]
                .sort_values("period_dt")
                .groupby(DEV_KEY)
                .apply(_slope))
    slopes = slopes_s.dropna().to_dict()

    slope_df = (
        pd.DataFrame.from_dict(slopes, orient="index", columns=["slope_days_per_day"])
        .reset_index().rename(columns={"index": DEV_KEY})
        .sort_values("slope_days_per_day")
    )

    print("=" * 60)
    print(f"MTBF AGING TREND - {DEVICE_TYPE}  (quarterly periods)")
    print("=" * 60)
    n_deg = (slope_df["slope_days_per_day"] < 0).sum()
    n_imp = (slope_df["slope_days_per_day"] > 0).sum()
    print(f"Devices analysed     : {len(slope_df)}")
    print(f"Degrading (slope < 0): {n_deg} ({n_deg/max(len(slope_df),1)*100:.1f}%)")
    print(f"Improving (slope > 0): {n_imp}")
    print("\nFastest Degrading Devices:")
    _deg_disp = slope_df.head(10).copy()
    if _id_map is not None:
        _deg_disp.insert(0, DEV_ID_COL, _deg_disp[DEV_KEY].map(_id_map))
    print(_deg_disp.round(4).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(
        f"{DEVICE_TYPE} - MTBF Aging Trend / Degradation Detection",
        fontsize=13, fontweight="bold"
    )

    pal = sns.color_palette("tab10", TOP_N_AGING)
    for i, dev in enumerate(top_devs_age):
        sub = period_mtbf[period_mtbf[DEV_KEY] == dev].sort_values("period_dt")
        _dev_lbl = str(_id_map.get(dev, dev)) if _id_map is not None else str(dev)
        axes[0].plot(sub["period_dt"], sub["TBF_days"], "o-", ms=5, lw=1.5,
                     label=_dev_lbl, color=pal[i])
        if dev in slopes and len(sub) >= 3:
            x_d   = (sub["period_dt"] - sub["period_dt"].min()).dt.days.values.astype(float)
            y_hat = slopes[dev] * x_d + sub["TBF_days"].iloc[0]
            axes[0].plot(sub["period_dt"], y_hat, "--", lw=1, color=pal[i], alpha=0.5)

    axes[0].set(title="Rolling MTBF per Device (solid=actual, dashed=trend)",
                xlabel="Quarter", ylabel="MTBF (days)")
    axes[0].legend(fontsize=7, ncol=2)
    axes[0].tick_params(axis="x", rotation=30)

    _top_slope = pd.concat([slope_df.head(10), slope_df.tail(10)]).drop_duplicates()
    _c2 = ["#E53935" if s < 0 else "#2E7D32" for s in _top_slope["slope_days_per_day"]]
    axes[1].barh(range(len(_top_slope)), _top_slope["slope_days_per_day"].values,
                 color=_c2, alpha=0.85)
    axes[1].set_yticks(range(len(_top_slope)))
    _e_labels = ([str(_id_map.get(d, d)) for d in _top_slope[DEV_KEY]]
                 if _id_map is not None else [str(d) for d in _top_slope[DEV_KEY]])
    axes[1].set_yticklabels(_e_labels, fontsize=8)
    axes[1].axvline(0, color="black", lw=1)
    axes[1].set(title="MTBF Trend Slope per Device\nRed=Degrading  Green=Improving",
                xlabel="MTBF slope (days/day)")
    axes[1].legend(
        handles=[_P2(color="#E53935", label="Degrading (MTBF falling)"),
                 _P2(color="#2E7D32", label="Improving (MTBF rising)")],
        fontsize=8
    )

    plt.tight_layout()
    plt.show()

    DEGRADING_DEVICES = slope_df[slope_df["slope_days_per_day"] < 0][DEV_KEY].tolist()
    print(f"\nDegrading devices: {len(DEGRADING_DEVICES)} stored in DEGRADING_DEVICES.")
else:
    print("[SKIP] MTBF aging trend - insufficient TBF data.")
    DEGRADING_DEVICES = []

In [ ]:
# ── Cell 7 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Fitter (Overall) ─────────────────────────────────────────────
if len(df_tbf) == 0:
    print("[SKIP] No TBF data.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — Kaplan-Meier Fitter (Overall, OOS-based)", fontsize=14, fontweight="bold")

    # Overall KM
    kmf_all = KaplanMeierFitter()
    kmf_all.fit(df_tbf["TBF_days"], event_observed=df_tbf["event"], label=f"All {DEVICE_TYPE} (OOS events)")
    kmf_all.plot_survival_function(ax=axes[0], ci_show=True, color="#7B1FA2")
    median_all = kmf_all.median_survival_time_
    axes[0].axhline(0.5, color="red", ls="--", lw=1, label=f"50% at {median_all:.1f}d")
    axes[0].set(title=f"KM Survival — All {DEVICE_TYPE} (median TBF={median_all:.1f}d)",
                xlabel="Days Between Failures", ylabel="Survival Probability")
    axes[0].legend()

    # TBF distribution
    axes[1].hist(df_tbf["TBF_days"].clip(upper=df_tbf["TBF_days"].quantile(0.99)),
                 bins=60, color="#7B1FA2", alpha=0.75, edgecolor="white")
    axes[1].set(title="Time Between Failures Distribution", xlabel="TBF (days)", ylabel="Count")
    axes[1].axvline(median_all, color="red", ls="--", lw=1.5, label=f"Median={median_all:.1f}d")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print(f"Overall KM median TBF: {median_all:.2f} days")


In [ ]:
# ── Cell 8 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Survival Analysis by Fault Code ──────────────────────────────
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP] No TBF data or fault codes.")
else:
    from joblib import Parallel, delayed as _jdel

    def _fit_km_fc(fc, _tbf, _fc_col):
        sub = _tbf[_tbf[_fc_col] == fc]
        if len(sub) < 5:
            return fc, None, None
        kmf = KaplanMeierFitter()
        kmf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
        return fc, kmf, kmf.median_survival_time_

    _km_fits = Parallel(n_jobs=-1, prefer="threads")(
        _jdel(_fit_km_fc)(fc, df_tbf, FC_COL) for fc in TOP10_FC
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Kaplan-Meier by Fault Code (Top {TOP_N})",
                 fontsize=14, fontweight="bold")
    pal = sns.color_palette("tab10", len(TOP10_FC))
    km_medians = {}

    for i, (fc, kmf, med) in enumerate(_km_fits):
        if kmf is None:
            continue
        kmf.plot_survival_function(ax=axes[0], ci_show=False, color=pal[i], lw=1.5)
        km_medians[fc] = med

    axes[0].axhline(0.5, color="black", ls=":", lw=1)
    axes[0].set(title="KM Survival by Fault Code", xlabel="Days", ylabel="Survival Probability")
    axes[0].legend(fontsize=7, ncol=2)

    med_df = pd.Series(km_medians).sort_values()
    colors = [pal[TOP10_FC.index(fc)] if fc in TOP10_FC else "gray" for fc in med_df.index]
    axes[1].barh(range(len(med_df)), med_df.values, color=colors, alpha=0.85)
    axes[1].set_yticks(range(len(med_df)))
    axes[1].set_yticklabels([f"FC={fc}" for fc in med_df.index], fontsize=9)
    axes[1].set(title="Median TBF per Fault Code (days)", xlabel="Median TBF (days)")

    plt.tight_layout()
    plt.show()

    if len(TOP10_FC) >= 2:
        fc1, fc2 = TOP10_FC[0], TOP10_FC[1]
        s1 = df_tbf[df_tbf[FC_COL] == fc1]
        s2 = df_tbf[df_tbf[FC_COL] == fc2]
        if len(s1) >= 5 and len(s2) >= 5:
            lr = logrank_test(s1["TBF_days"], s2["TBF_days"],
                              event_observed_A=s1["event"], event_observed_B=s2["event"])
            print(f"\nLog-rank test FC={fc1} vs FC={fc2}:")
            print(f"  p-value = {lr.p_value:.6f}  (significant if < 0.05)")

# ── MLflow: Kaplan-Meier results ─────────────────────────────────────────────
try:
    with mlflow.start_run(run_name="kaplan_meier_fc") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "nonparametric_km"})
        mlflow.log_params({"n_fc": len(TOP10_FC), "min_samples": 5,
                           "estimator": "KaplanMeier"})
        for _fc, _kmf, _med in _km_fits:
            if _kmf is not None and _med == _med:  # not NaN
                mlflow.log_metric(f"km_median_tbf_fc_{_fc}", float(_med))
    print(f"[MLflow] KM run: {_run.info.run_id}")
except Exception as _mle:
    print(f"[MLflow] KM log skipped: {_mle}")

In [ ]:
# ── Cell 9 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Analysis for a Specific Device ───────────────────────────────
if len(df_tbf) == 0 or SELECTED_DEVICE is None:
    print("[SKIP]")
else:
    sub_dev = df_tbf[df_tbf[DEV_KEY] == SELECTED_DEVICE]
    print(f"Device {SELECTED_DEVICE}: {len(sub_dev):,} TBF intervals")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — KM for Device {SELECTED_DEVICE} (highest-failure device)",
                 fontsize=13, fontweight="bold")

    # Overall KM for selected device
    kmf_dev = KaplanMeierFitter()
    kmf_dev.fit(sub_dev["TBF_days"], sub_dev["event"], label=f"Device {SELECTED_DEVICE}")
    kmf_dev.plot_survival_function(ax=axes[0], ci_show=True, color="#E53935")
    med_dev = kmf_dev.median_survival_time_
    axes[0].axhline(0.5, color="black", ls="--", lw=1, label=f"50% at {med_dev:.1f}d")
    axes[0].set(title=f"KM Survival — Device {SELECTED_DEVICE}  median={med_dev:.1f}d",
                xlabel="Days", ylabel="Survival Probability")
    axes[0].legend()

    # KM per fault code for selected device
    fc_in_dev = sub_dev[FC_COL].value_counts().head(5).index.tolist() if FC_COL else []
    pal = sns.color_palette("tab10", 5)
    for i, fc in enumerate(fc_in_dev):
        s = sub_dev[sub_dev[FC_COL] == fc]
        if len(s) >= 3:
            KaplanMeierFitter().fit(s["TBF_days"], s["event"],
                                    label=f"FC={fc}").plot_survival_function(
                ax=axes[1], ci_show=False, color=pal[i])
    axes[1].set(title=f"KM by Fault Code — Device {SELECTED_DEVICE}",
                xlabel="Days", ylabel="Survival Prob")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 10 ──────────────────────────────────────────────────────────
# ── Weibull Fitting for Different Fault Codes ─────────────────────────────────
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    from joblib import Parallel, delayed as _jdel

    def _fit_wb_fc(fc, _tbf, _fc_col):
        sub = _tbf[_tbf[_fc_col] == fc]
        if len(sub) < 5:
            return fc, len(sub), None, "too few rows"
        try:
            wf = WeibullFitter()
            wf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
            return fc, len(sub), wf, None
        except Exception as e:
            return fc, len(sub), None, str(e)[:40]

    _wb_fits = Parallel(n_jobs=-1, prefer="threads")(
        _jdel(_fit_wb_fc)(fc, df_tbf, FC_COL) for fc in TOP10_FC
    )

    wb_params = []
    for fc, cnt, wf, err in _wb_fits:
        if wf is not None:
            beta  = wf.rho_
            eta   = wf.lambda_ ** (-1/beta) if beta > 0 else float("nan")
            med   = wf.median_survival_time_
            ftype = ("Infant mortality (β<1)" if beta < 1
                     else "Random failures (β≈1)" if abs(beta-1) < 0.15
                     else "Wear-out failures (β>1)")
            wb_params.append({"FaultCode": fc, "Count": cnt,
                              "β (shape)": round(beta, 4), "η (scale,d)": round(eta, 2),
                              "Median_TBF_d": round(med, 2), "FailureType": ftype})
        else:
            wb_params.append({"FaultCode": fc, "Count": cnt,
                              "β (shape)": None, "η (scale,d)": None,
                              "Median_TBF_d": None, "FailureType": err})

    wb_df = pd.DataFrame(wb_params)
    print(f"{'='*70}")
    print(f"WEIBULL PARAMETERS — {DEVICE_TYPE} Top 10 Fault Codes")
    print(f"{'='*70}")
    print(wb_df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — Weibull Fitting by Fault Code (β Analysis)",
                 fontsize=13, fontweight="bold")

    wb_plot = wb_df.dropna(subset=["β (shape)"])
    colors = ["#E53935" if b < 1 else "#FF6F00" if abs(b-1) < 0.15 else "#1565C0"
              for b in wb_plot["β (shape)"]]
    axes[0].barh(range(len(wb_plot)), wb_plot["β (shape)"].values, color=colors, alpha=0.85)
    axes[0].axvline(1.0, color="black", ls="--", lw=1.5, label="β=1 (Random)")
    axes[0].set_yticks(range(len(wb_plot)))
    axes[0].set_yticklabels([f"FC={fc}" for fc in wb_plot["FaultCode"]], fontsize=9)
    axes[0].set(title="β (Shape Parameter) per Fault Code", xlabel="β")
    from matplotlib.patches import Patch as _P
    axes[0].legend(handles=[_P(color="#E53935", label="β<1 Infant"),
                             _P(color="#FF6F00", label="β≈1 Random"),
                             _P(color="#1565C0", label="β>1 Wear-out")])

    pal = sns.color_palette("tab10", len(TOP10_FC))
    for i, (fc, cnt, wf, err) in enumerate(_wb_fits):
        if wf is None:
            continue
        wf.plot_survival_function(ax=axes[1], ci_show=False, color=pal[i], lw=1.5)
    axes[1].set(title="Weibull Survival Curves by Fault Code", xlabel="Days", ylabel="Survival Prob")
    axes[1].legend(fontsize=7, ncol=2)

    plt.tight_layout()
    plt.show()

# ── MLflow: Weibull results ──────────────────────────────────────────────────
try:
    with mlflow.start_run(run_name="weibull_fitter_fc") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "parametric_weibull"})
        mlflow.log_params({"n_fc": len(TOP10_FC), "estimator": "WeibullFitter"})
        for _fc, _cnt, _wf, _err in _wb_fits:
            if _wf is not None:
                mlflow.log_metric(f"weibull_beta_fc_{_fc}",
                                  float(_wf.rho_))
                mlflow.log_metric(f"weibull_median_fc_{_fc}",
                                  float(_wf.median_survival_time_))
        _wcsv = os.path.join(OUT_DIR, "weibull_params.csv")
        wb_df.to_csv(_wcsv, index=False)
        mlflow.log_artifact(_wcsv, artifact_path="tables")
    print(f"[MLflow] Weibull run: {_run.info.run_id}")
except Exception as _mle:
    print(f"[MLflow] Weibull log skipped: {_mle}")

In [ ]:
# ── Cell 10b — Optimal PM Interval from Weibull Parameters ───────────────────
# For each fault code Weibull fit, compute the PM interval T* that achieves
# a target reliability R:  T* = eta x (-ln(R))^(1/beta)
# beta > 1 (wear-out): schedule PM at T* to prevent failure.
# beta <= 1 (random/infant): PM has little value — focus on MTTR instead.
R_TARGETS = [0.99, 0.95, 0.90]

_wb_source = None
if "wb_df" in dir() and len(wb_df) > 0:
    _wb_source = wb_df.copy()
elif "wb_params" in dir() and len(wb_params) > 0:
    _wb_source = pd.DataFrame(wb_params)

if _wb_source is not None:
    pm_rows = []
    _beta_col = next((c for c in _wb_source.columns
                      if "beta" in c.lower() or "shape" in c.lower()), None)
    _eta_col  = next((c for c in _wb_source.columns
                      if "eta" in c.lower() or "scale" in c.lower()), None)
    _fc_col   = next((c for c in _wb_source.columns
                      if "fault" in c.lower() or "code" in c.lower() or c == "FaultCode"), None)

    if _beta_col and _eta_col and _fc_col:
        for _, row in _wb_source.iterrows():
            try:
                beta = float(row[_beta_col]); eta = float(row[_eta_col])
                fc   = row[_fc_col]
            except (TypeError, ValueError):
                continue
            if beta <= 0 or eta <= 0:
                continue
            r = {
                "FaultCode"  : fc,
                "Count"      : int(row.get("Count", 0)),
                "beta"       : round(beta, 3),
                "eta_days"   : round(eta, 2),
                "FailureMode": ("Infant" if beta < 0.85 else "Random" if beta < 1.15 else "Wear-out"),
            }
            for rt in R_TARGETS:
                t_star = eta * ((-np.log(rt)) ** (1.0 / beta))
                r[f"PM@{int(rt*100)}pct_days"] = round(t_star, 1)
            pm_rows.append(r)
    else:
        print(f"[WARN] Could not identify beta/eta/FC columns. Available: {list(_wb_source.columns)}")

    if pm_rows:
        pm_df = pd.DataFrame(pm_rows)
        print("=" * 75)
        print(f"OPTIMAL PM INTERVAL - {DEVICE_TYPE}  [T* = eta x (-ln(R))^(1/beta)]")
        print("=" * 75)
        print(pm_df.to_string(index=False))

        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        fig.suptitle(f"{DEVICE_TYPE} - Optimal PM Interval per Fault Code",
                     fontsize=13, fontweight="bold")

        x = np.arange(len(pm_df)); w = 0.25
        axes[0].bar(x - w, pm_df["PM@99pct_days"], w, label="99% reliability",
                    color="#1565C0", alpha=0.85)
        axes[0].bar(x,     pm_df["PM@95pct_days"], w, label="95% reliability",
                    color="#FF6F00", alpha=0.85)
        axes[0].bar(x + w, pm_df["PM@90pct_days"], w, label="90% reliability",
                    color="#E53935", alpha=0.85)
        axes[0].set_xticks(x)
        axes[0].set_xticklabels([f"FC={fc}" for fc in pm_df["FaultCode"]],
                                 rotation=45, fontsize=8)
        axes[0].set(title="PM Interval by Reliability Target", ylabel="PM Interval (days)")
        axes[0].legend()

        _sc = axes[1].scatter(pm_df["beta"], pm_df["PM@95pct_days"],
                              c=pm_df["Count"], cmap="Blues", s=80,
                              edgecolors="gray", alpha=0.85)
        plt.colorbar(_sc, ax=axes[1], label="Failure Count")
        axes[1].axvline(1.0, color="black", ls="--", lw=1, label="beta=1 (Random)")
        for _, r in pm_df.iterrows():
            axes[1].annotate(str(r["FaultCode"]), (r["beta"], r["PM@95pct_days"]),
                             fontsize=7, xytext=(4, 2), textcoords="offset points")
        axes[1].set(
            title="PM Interval @95% vs beta Shape\nbeta>1 Wear-out: shorter PM; beta<1 Infant: PM less effective",
            xlabel="beta (Weibull shape)", ylabel="PM Interval @95% (days)"
        )
        axes[1].legend(fontsize=8)

        plt.tight_layout()
        plt.show()

        print("\nInterpretation:")
        print("  beta > 1 (wear-out) : PM interval = T* — schedule BEFORE failure.")
        print("  beta ~ 1 (random)   : PM has minimal impact — improve MTTR instead.")
        print("  beta < 1 (infant)   : Replace at installation; PM does not help.")
    else:
        print("[SKIP] No valid Weibull parameters - check wb_df column names.")
else:
    print("[SKIP] Weibull parameters not found - run Weibull Fitting cell first.")


In [ ]:
# ── Cell 11 ──────────────────────────────────────────────────────────
# ── Weibull Fitting for the Selected Device ────────────────────────────────────
if len(df_tbf) == 0 or SELECTED_DEVICE is None:
    print("[SKIP]")
else:
    sub_dev = df_tbf[df_tbf[DEV_KEY] == SELECTED_DEVICE]
    if len(sub_dev) < 5:
        print(f"[SKIP] Only {len(sub_dev)} TBF records for device {SELECTED_DEVICE}.")
    else:
        wf = WeibullFitter()
        wf.fit(sub_dev["TBF_days"], sub_dev["event"])
        beta  = wf.rho_
        eta   = wf.lambda_ ** (-1/beta) if beta > 0 else float("nan")
        med   = wf.median_survival_time_
        ftype = ("Infant mortality (β<1)" if beta < 1
                 else "Random failures (β≈1)" if abs(beta-1) < 0.15
                 else "Wear-out failures (β>1)")

        print(f"Weibull for Device {SELECTED_DEVICE}:")
        print(f"  β (shape) = {beta:.4f}  ->  {ftype}")
        print(f"  η (scale) = {eta:.2f} days")
        print(f"  Median TBF = {med:.2f} days")

        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        fig.suptitle(f"{DEVICE_TYPE} — Weibull for Device {SELECTED_DEVICE}  β={beta:.3f}  [{ftype.split('(')[0].strip()}]",
                     fontsize=13, fontweight="bold")

        wf.plot_survival_function(ax=axes[0], color="#E53935")
        axes[0].set(title=f"Weibull Survival  β={beta:.3f}", xlabel="Days")

        wf.plot_cumulative_hazard(ax=axes[1], color="#1565C0")
        axes[1].set(title="Cumulative Hazard", xlabel="Days")

        tbf_clip = sub_dev["TBF_days"].clip(upper=sub_dev["TBF_days"].quantile(0.99))
        axes[2].hist(tbf_clip, bins=40, density=True, color="#FFA726", alpha=0.75, edgecolor="white", label="TBF")
        if beta > 0 and not np.isnan(eta):
            x = np.linspace(0.1, tbf_clip.max(), 300)
            axes[2].plot(x, weibull_min.pdf(x, beta, scale=eta), "b-", lw=2, label="Weibull fit")
        axes[2].set(title="TBF Distribution + Weibull Fit", xlabel="Days", ylabel="Density")
        axes[2].legend()

        plt.tight_layout()
        plt.show()


In [ ]:
# ── Cell 12 ──────────────────────────────────────────────────────────
# ── Weibull Failure Prediction per Fault Code ──────────────────────────────────
# Statistical prediction of expected time to next failure using Weibull distribution
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    pred_rows = []
    for fc in TOP10_FC:
        sub = df_tbf[df_tbf[FC_COL] == fc]
        if len(sub) < 5: continue
        try:
            wf = WeibullFitter()
            wf.fit(sub["TBF_days"], sub["event"])
            beta = wf.rho_
            eta  = wf.lambda_ ** (-1/beta) if beta > 0 else float("nan")
            # Weibull predictions: prob of surviving t=7, 14, 30, 60 days
            surv_7d  = float(wf.survival_function_at_times([7]).iloc[0])   if not np.isnan(eta) else float("nan")
            surv_30d = float(wf.survival_function_at_times([30]).iloc[0])  if not np.isnan(eta) else float("nan")
            med      = wf.median_survival_time_
            pred_rows.append({"FaultCode": fc, "Count": len(sub),
                              "β": round(beta, 3), "Median_TBF_d": round(med, 1),
                              "P(survive>7d)": round(surv_7d, 4),
                              "P(survive>30d)": round(surv_30d, 4),
                              "P(fail<=7d)": round(1 - surv_7d, 4),
                              "P(fail<=30d)": round(1 - surv_30d, 4)})
        except Exception: pass

    if pred_rows:
        pred_df = pd.DataFrame(pred_rows)
        print(f"{'='*75}")
        print(f"WEIBULL FAILURE PREDICTION — {DEVICE_TYPE} (probability of failure within horizon)")
        print(f"{'='*75}")
        print(pred_df.to_string(index=False))

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(f"{DEVICE_TYPE} — Weibull Failure Prediction per Fault Code",
                     fontsize=13, fontweight="bold")

        pal = sns.color_palette("tab10", len(pred_df))
        axes[0].barh(range(len(pred_df)), pred_df["P(fail<=7d)"], color="#E53935", alpha=0.85)
        axes[0].set_yticks(range(len(pred_df)))
        axes[0].set_yticklabels([f"FC={r}" for r in pred_df["FaultCode"]], fontsize=9)
        axes[0].set(title="P(Failure within 7 days) per Fault Code", xlabel="Probability")

        axes[1].barh(range(len(pred_df)), pred_df["P(fail<=30d)"], color="#FF6F00", alpha=0.85)
        axes[1].set_yticks(range(len(pred_df)))
        axes[1].set_yticklabels([f"FC={r}" for r in pred_df["FaultCode"]], fontsize=9)
        axes[1].set(title="P(Failure within 30 days) per Fault Code", xlabel="Probability")

        plt.tight_layout()
        plt.show()


In [ ]:
# ── Cell 13 ──────────────────────────────────────────────────────────
# ── Cox Proportional Hazard Model ─────────────────────────────────────────────
# Fault codes as one-hot covariates; additional: is_chargeable, priority, time_to_resolve
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    # One-hot encode top-10 fault codes
    df_cox = df_tbf[[DEV_KEY, FC_COL, "TBF_days", "event"]].copy()
    for fc in TOP10_FC:
        df_cox[f"FC_{fc}"] = (df_cox[FC_COL] == fc).astype(int)

    # Add numeric covariates if available
    cov_candidates = ["is_chargeable", "priority", "time_to_resolve_minutes", "is_major_incident"]
    extra_cov = [c for c in cov_candidates if c in df_tbf.columns]
    # df_cox is a column-subset of df_tbf with the same row order — direct assign, no merge needed
    for col in cov_candidates:
        if col in df_tbf.columns:
            df_cox[col] = df_tbf[col].values

    fc_cols = [f"FC_{fc}" for fc in TOP10_FC if f"FC_{fc}" in df_cox.columns]
    # Drop all-zero FC columns — NaN fault codes produce all-zero one-hot cols
    # because (nan == nan) is False, so no row ever gets a 1.
    fc_cols = [c for c in fc_cols if df_cox[c].sum() > 0]
    # Drop FC cols where the fault code is too rare (<1% of rows)
    fc_cols = [c for c in fc_cols if df_cox[c].mean() >= 0.01]
    # Drop one FC as reference category — fault codes are mutually exclusive per row
    # (each TBF interval has exactly one FC), so all columns sum to 1: perfect multicollinearity.
    # Dropping one breaks the rank deficiency just like drop_first=True in get_dummies.
    if len(fc_cols) > 1:
        _ref_fc = fc_cols[0]  # most frequent FC (TOP10_FC is freq-sorted)
        fc_cols = fc_cols[1:]
        print(f"  Reference FC dropped: {_ref_fc}")
    num_cov = [c for c in cov_candidates if c in df_cox.columns]
    # Coerce to float — columns may arrive as decimal.Decimal or object dtype
    for _col in num_cov:
        df_cox[_col] = pd.to_numeric(df_cox[_col], errors="coerce")
    # Drop zero-variance numeric covariates (constant columns also cause singularity)
    num_cov = [c for c in num_cov if df_cox[c].std(skipna=True) > 0]
    all_cov = fc_cols + num_cov
    df_cox[all_cov] = df_cox[all_cov].fillna(0)

    # Scale numeric covariates
    if num_cov:
        sc = StandardScaler()
        df_cox[num_cov] = sc.fit_transform(df_cox[num_cov])

    cox_data = df_cox[["TBF_days", "event"] + all_cov].dropna()
    print(f"  Cox PH: {len(cox_data):,} rows, {len(all_cov)} covariates")
    cph = None
    for _pen in [0.1, 1.0, 5.0, 10.0, 50.0, 100.0]:
        try:
            _cph = CoxPHFitter(penalizer=_pen)
            _cph.fit(cox_data, duration_col="TBF_days", event_col="event")
            cph = _cph
            print(f"  Converged with penalizer={_pen}")
            break
        except Exception as _e:
            print(f"  penalizer={_pen} → {type(_e).__name__}, retrying...")
    if cph is None:
        print("[SKIP] Cox PH failed to converge — data may be too sparse or collinear")
    else:

        print("Cox Proportional Hazard Model Summary:")
        print(cph.summary[["coef", "exp(coef)", "p", "coef lower 95%", "coef upper 95%"]].round(4).to_string())
        print(f"\nConcordance index: {cph.concordance_index_:.4f}")
        print(f"Log-likelihood    : {cph.log_likelihood_:.4f}")

        # ── Plot Cox PH Model results ─────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        fig.suptitle(f"{DEVICE_TYPE} — Cox PH Model  CI={cph.concordance_index_:.3f}",
                     fontsize=13, fontweight="bold")

        coef = cph.summary["coef"].sort_values()
        colors = ["#E53935" if v > 0 else "#1565C0" for v in coef.values]
        axes[0].barh(range(len(coef)), coef.values, color=colors, alpha=0.85)
        axes[0].set_yticks(range(len(coef)))
        axes[0].set_yticklabels([c[:20] for c in coef.index], fontsize=8)
        axes[0].axvline(0, color="black", lw=1)
        axes[0].set(title="Cox Coefficients (log Hazard Ratios)\nRed=increases risk, Blue=decreases risk",
                    xlabel="Coefficient")

        # Partial effect of most influential feature
        top_feat = coef.abs().idxmax()
        try:
            cph.plot_partial_effects_on_outcome(top_feat, values=[-1, 0, 1], ax=axes[1])
            axes[1].set(title=f"Partial Effect of '{top_feat}'")
        except Exception as e:
            axes[1].text(0.5, 0.5, f"Plot error:\n{e}", ha="center", va="center",
                        transform=axes[1].transAxes, fontsize=9)
            axes[1].set_title(f"Partial Effect — {top_feat}")

        plt.tight_layout()
        plt.show()

# ── MLflow: Cox PH (Champion) ────────────────────────────────────────────────
try:
    with mlflow.start_run(run_name="cox_ph_model") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "cox_ph", "champion": "true"})
        try:
            _cx_params = {
                "penalizer":    _pen if cph is not None else "failed",
                "n_covariates": len(all_cov),
                "n_rows":       len(cox_data),
            }
        except NameError:
            _cx_params = {"penalizer": "not_run", "n_covariates": 0, "n_rows": 0}
        mlflow.log_params(_cx_params)
        if cph is not None:
            CHAMPION_CI     = float(cph.concordance_index_)
            CHAMPION_RUN_ID = _run.info.run_id
            mlflow.log_metric("concordance_index", CHAMPION_CI)
            mlflow.log_metric("log_likelihood",    float(cph.log_likelihood_))
            _cox_pkl = os.path.join(OUT_DIR, "cox_ph_model.pkl")
            with open(_cox_pkl, "wb") as _f:
                pickle.dump(cph, _f)
            mlflow.log_artifact(_cox_pkl, artifact_path="model")
            _hr = np.exp(cph.params_).rename("hazard_ratio").reset_index()
            _hr.columns = ["covariate", "hazard_ratio"]
            _hrcsv = os.path.join(OUT_DIR, "cox_hazard_ratios.csv")
            _hr.to_csv(_hrcsv, index=False)
            mlflow.log_artifact(_hrcsv, artifact_path="tables")
    _ci_s = f"CI={CHAMPION_CI:.4f}" if CHAMPION_CI is not None else "no convergence"
    print(f"[MLflow] Cox PH run: {_run.info.run_id}  {_ci_s}")
except Exception as _mle:
    print(f"[MLflow] Cox PH log skipped: {_mle}")

In [ ]:
# ── Cell 14 ──────────────────────────────────────────────────────────
# ── Failure Rate Over Time per Fault Code ─────────────────────────────────────
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    fig, ax = plt.subplots(figsize=(16, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Failure Rate Over Time per Fault Code (Top {min(5, len(TOP10_FC))})",
                 fontsize=13, fontweight="bold")

    pal = sns.color_palette("tab10", len(TOP10_FC))
    for i, fc in enumerate(TOP10_FC[:5]):
        sub = df[df[FC_COL] == fc].copy()
        if len(sub) == 0: continue
        monthly = sub.groupby(sub[DATE_COL].dt.to_period("M")).size()
        ax.plot(range(len(monthly)), monthly.values, marker="o", ms=4, lw=1.5,
                label=f"FC={fc}", color=pal[i])

    # x-axis labels
    _sample_fc = TOP10_FC[0]
    _sample_sub = df[df[FC_COL] == _sample_fc]
    if len(_sample_sub) > 0:
        _months = _sample_sub.groupby(_sample_sub[DATE_COL].dt.to_period("M")).size().index
        step = max(1, len(_months) // 8)
        ax.set_xticks(range(0, len(_months), step))
        ax.set_xticklabels([str(m) for m in _months[::step]], rotation=45, fontsize=8)

    ax.set(title="Monthly Failure Count per Fault Code", xlabel="Month", ylabel="Failure Count")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 15 ──────────────────────────────────────────────────────────
# ── Hazard Function Estimation per Fault Code (Nelson-Aalen) ──────────────────
if len(df_tbf) == 0 or not TOP10_FC:
    print("[SKIP]")
else:
    from joblib import Parallel, delayed as _jdel

    def _fit_naf_fc(fc, _tbf, _fc_col):
        sub = _tbf[_tbf[_fc_col] == fc]
        if len(sub) < 5:
            return None
        naf = NelsonAalenFitter()
        naf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
        return naf

    _naf_fits = Parallel(n_jobs=-1, prefer="threads")(
        _jdel(_fit_naf_fc)(fc, df_tbf, FC_COL) for fc in TOP10_FC
    )

    fig, ax = plt.subplots(figsize=(16, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Nelson-Aalen Hazard Function per Fault Code (Top {TOP_N})",
                 fontsize=13, fontweight="bold")
    pal = sns.color_palette("tab10", len(TOP10_FC))

    for i, naf in enumerate(_naf_fits):
        if naf is None:
            continue
        naf.plot_cumulative_hazard(ax=ax, ci_show=False, color=pal[i], lw=1.5)

    ax.set(title="Cumulative Hazard Function per Fault Code", xlabel="Days Between Failures")
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

# ── MLflow: Nelson-Aalen results ─────────────────────────────────────────────
try:
    _naf_count = sum(1 for _n in _naf_fits if _n is not None)
    with mlflow.start_run(run_name="nelson_aalen_fc") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "nonparametric_na"})
        mlflow.log_params({"n_fitters": _naf_count, "estimator": "NelsonAalen"})
    print(f"[MLflow] Nelson-Aalen run: {_run.info.run_id}  ({_naf_count} fitters)")
except Exception as _mle:
    print(f"[MLflow] Nelson-Aalen log skipped: {_mle}")

In [ ]:
# ── Cell 16 ──────────────────────────────────────────────────────────
# ── Method 1: Multi-Factor Risk Ranking ───────────────────────────────────────
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    dev_stats = df_tbf.groupby(DEV_KEY).agg(
        failure_count=("TBF_days", "count"),
        avg_tbf=("TBF_days", "mean"),
        min_tbf=("TBF_days", "min"),
    ).reset_index()
    dev_stats["MTBF_days"] = dev_stats["avg_tbf"]

    def _norm(s): rng = s.max()-s.min(); return (s-s.min())/rng if rng>0 else s*0
    dev_stats["risk_score"] = (
        _norm(dev_stats["failure_count"]) +
        (1 - _norm(dev_stats["MTBF_days"])) +
        (1 - _norm(dev_stats["min_tbf"]))
    ) / 3.0

    top10_risk = dev_stats.nlargest(10, "risk_score")
    print(f"{'='*60}")
    print(f"Method 1: Multi-Factor Risk Ranking — Top 10 {DEVICE_TYPE} Devices")
    print(f"{'='*60}")
    _t15 = top10_risk.copy()
    if _id_map is not None:
        _t15.insert(0, DEV_ID_COL, _t15[DEV_KEY].map(_id_map))
    _c15 = ([DEV_ID_COL, DEV_KEY] if _id_map is not None else [DEV_KEY]) + ["failure_count", "MTBF_days", "min_tbf", "risk_score"]
    print(_t15[_c15].round(3).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — Method 1: Multi-Factor Risk Ranking", fontsize=13, fontweight="bold")

    axes[0].barh(range(len(top10_risk)), top10_risk["risk_score"].values, color="#E53935", alpha=0.85)
    axes[0].set_yticks(range(len(top10_risk)))
    _yl15 = ([str(_id_map.get(d, d)) for d in top10_risk[DEV_KEY]]
             if _id_map is not None else [str(d) for d in top10_risk[DEV_KEY]])
    axes[0].set_yticklabels(_yl15, fontsize=9)
    axes[0].set(title="Top 10 High-Risk Devices (Multi-Factor Score)", xlabel="Risk Score")

    axes[1].scatter(dev_stats["MTBF_days"], dev_stats["failure_count"],
                    c=dev_stats["risk_score"], cmap="RdYlGn_r", alpha=0.6, s=30, edgecolors="none")
    axes[1].set(title="Failure Count vs MTBF (Risk color)", xlabel="MTBF (days)", ylabel="Failure Count")
    plt.colorbar(axes[1].collections[0], ax=axes[1], label="Risk Score")

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 17 ──────────────────────────────────────────────────────────
# ── MTBF-Based Risk Ranking ───────────────────────────────────────────────────
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    # Risk score = 1/MTBF (normalized)
    dev_stats["mtbf_risk"] = _norm(1.0 / dev_stats["MTBF_days"].replace(0, np.nan).fillna(dev_stats["MTBF_days"].max()))
    top10_mtbf = dev_stats.nlargest(10, "mtbf_risk")

    print(f"{'='*60}")
    print(f"MTBF-Based Risk Ranking — Top 10 {DEVICE_TYPE} Devices")
    print(f"{'='*60}")
    _t16 = top10_mtbf.copy()
    if _id_map is not None:
        _t16.insert(0, DEV_ID_COL, _t16[DEV_KEY].map(_id_map))
    _c16 = ([DEV_ID_COL, DEV_KEY] if _id_map is not None else [DEV_KEY]) + ["failure_count", "MTBF_days", "mtbf_risk"]
    print(_t16[_c16].round(3).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"{DEVICE_TYPE} — MTBF-Based Risk Ranking", fontsize=13, fontweight="bold")

    axes[0].hist(dev_stats["MTBF_days"].clip(upper=dev_stats["MTBF_days"].quantile(0.99)),
                 bins=40, color="#1565C0", alpha=0.8, edgecolor="white")
    med_mtbf = dev_stats["MTBF_days"].median()
    axes[0].axvline(med_mtbf, color="red", ls="--", lw=1.5, label=f"Median={med_mtbf:.1f}d")
    axes[0].set(title="MTBF Distribution Across All Devices", xlabel="MTBF (days)", ylabel="Devices")
    axes[0].legend()

    axes[1].barh(range(len(top10_mtbf)), top10_mtbf["MTBF_days"].values, color="#FF6F00", alpha=0.85)
    axes[1].set_yticks(range(len(top10_mtbf)))
    _yl16 = ([str(_id_map.get(d, d)) for d in top10_mtbf[DEV_KEY]]
             if _id_map is not None else [str(d) for d in top10_mtbf[DEV_KEY]])
    axes[1].set_yticklabels(_yl16, fontsize=9)
    axes[1].axvline(med_mtbf, color="red", ls="--", lw=1, label=f"Fleet median={med_mtbf:.1f}d")
    axes[1].set(title="Shortest MTBF Devices (highest failure frequency)", xlabel="MTBF (days)")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 18 ──────────────────────────────────────────────────────────
# ── Device + Fault Code Risk — Failure Reasons in Depth ──────────────────────
# Which fault codes are most problematic per device?
if len(df_tbf) == 0 or not FC_COL:
    print("[SKIP]")
else:
    # MTBF per (Device, FaultCode) pair
    dev_fc_stats = df_tbf.groupby([DEV_KEY, FC_COL]).agg(
        count=("TBF_days", "count"),
        avg_tbf=("TBF_days", "mean"),
    ).reset_index()
    dev_fc_stats["risk_score"] = _norm(dev_fc_stats["count"]) + (1 - _norm(dev_fc_stats["avg_tbf"]))

    top20_pairs = dev_fc_stats.nlargest(20, "risk_score")
    print(f"{'='*65}")
    print(f"Device + Fault Code Risk — Top 20 Pairs  ({DEVICE_TYPE})")
    print(f"{'='*65}")
    _t17 = top20_pairs.copy()
    if _id_map is not None:
        _t17.insert(0, DEV_ID_COL, _t17[DEV_KEY].map(_id_map))
    _c17 = ([DEV_ID_COL, DEV_KEY] if _id_map is not None else [DEV_KEY]) + [FC_COL, "count", "avg_tbf", "risk_score"]
    print(_t17[_c17].round(3).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f"{DEVICE_TYPE} — Device + Fault Code Risk (Failure Reasons per Device)",
                 fontsize=13, fontweight="bold")

    labels = [(f"{_id_map.get(r[DEV_KEY], r[DEV_KEY])}\nFC={r[FC_COL]}"
               if _id_map is not None else f"{r[DEV_KEY]}\nFC={r[FC_COL]}")
              for _, r in top20_pairs.iterrows()]
    axes[0].barh(range(len(top20_pairs)), top20_pairs["risk_score"].values, color="#E53935", alpha=0.85)
    axes[0].set_yticks(range(len(top20_pairs)))
    axes[0].set_yticklabels(labels, fontsize=7)
    axes[0].set(title="Top 20 Device+FaultCode Risk Pairs", xlabel="Risk Score")

    axes[1].barh(range(len(top20_pairs)), top20_pairs["avg_tbf"].values, color="#FF6F00", alpha=0.85)
    axes[1].set_yticks(range(len(top20_pairs)))
    axes[1].set_yticklabels(labels, fontsize=7)
    axes[1].set(title="Avg TBF per Device+FaultCode Pair (days)", xlabel="Avg TBF (days)")

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 19 ──────────────────────────────────────────────────────────
# ── Severity Levels — Device-Level Risk Ranking ───────────────────────────────
# Severity Score = Nelson-Aalen cumulative hazard + (1 - normalized MTBF)
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    top_devs = df_tbf[DEV_KEY].value_counts().nlargest(TOP_N).index.tolist()
    dev_hazard = {}
    for dev in top_devs:
        sub = df_tbf[df_tbf[DEV_KEY] == dev]
        if len(sub) >= 3:
            naf = NelsonAalenFitter()
            naf.fit(sub["TBF_days"], sub["event"])
            dev_hazard[dev] = float(naf.cumulative_hazard_.iloc[-1, 0])

    if dev_hazard:
        sev = pd.DataFrame.from_dict(dev_hazard, orient="index", columns=["cum_hazard"])
        sev.index.name = DEV_KEY; sev = sev.reset_index()
        sev = sev.merge(dev_stats[[DEV_KEY, "MTBF_days", "failure_count"]], on=DEV_KEY, how="left")
        sev["mtbf_norm"] = _norm(sev["MTBF_days"].fillna(sev["MTBF_days"].median()))
        sev["severity"]  = sev["cum_hazard"] + (1 - sev["mtbf_norm"])
        sev = sev.sort_values("severity", ascending=False)

        # Assign severity labels
        q75 = sev["severity"].quantile(0.75)
        q50 = sev["severity"].quantile(0.5)
        sev["severity_level"] = sev["severity"].apply(
            lambda x: "CRITICAL" if x >= q75 else ("HIGH" if x >= q50 else "MEDIUM"))

        print(f"{'='*60}")
        print(f"Severity Levels — {DEVICE_TYPE} Devices")
        print(f"{'='*60}")
        _t18 = sev.copy()
        if _id_map is not None:
            _t18.insert(0, DEV_ID_COL, _t18[DEV_KEY].map(_id_map))
        _c18 = ([DEV_ID_COL, DEV_KEY] if _id_map is not None else [DEV_KEY]) + ["failure_count", "MTBF_days", "cum_hazard", "severity", "severity_level"]
        print(_t18[_c18].round(3).to_string(index=False))

        fig, ax = plt.subplots(figsize=(12, 7))
        fig.suptitle(f"{DEVICE_TYPE} — Severity Levels: Device-Level Risk Ranking", fontsize=13, fontweight="bold")

        color_map = {"CRITICAL": "#E53935", "HIGH": "#FF6F00", "MEDIUM": "#1565C0"}
        colors = [color_map.get(lvl, "#999") for lvl in sev["severity_level"]]
        ax.barh(range(len(sev)), sev["severity"].values, color=colors, alpha=0.85)
        ax.set_yticks(range(len(sev)))
        _yl18 = ([str(_id_map.get(d, d)) for d in sev[DEV_KEY]]
                 if _id_map is not None else [str(d) for d in sev[DEV_KEY]])
        ax.set_yticklabels(_yl18, fontsize=8)
        ax.set(title="Device Severity Score (Hazard + 1/MTBF)", xlabel="Severity Score")
        from matplotlib.patches import Patch as _P
        ax.legend(handles=[_P(color="#E53935",label="CRITICAL"),
                           _P(color="#FF6F00",label="HIGH"),
                           _P(color="#1565C0",label="MEDIUM")])
        plt.tight_layout()
        plt.show()


In [ ]:
# ── Cell 20 ──────────────────────────────────────────────────────────
# ── Severity Analysis for Device + Fault Code Risk Ranking ────────────────────
# Nelson-Aalen per (Device, FaultCode) — severity heatmap
if len(df_tbf) == 0 or not FC_COL:
    print("[SKIP]")
else:
    top10_dev_sev = df_tbf[DEV_KEY].value_counts().nlargest(10).index.tolist()
    top5_fc_sev   = df_tbf[FC_COL].value_counts().nlargest(5).index.tolist()
    sev_pairs = []
    for dev in top10_dev_sev:
        for fc in top5_fc_sev:
            sub = df_tbf[(df_tbf[DEV_KEY] == dev) & (df_tbf[FC_COL] == fc)]
            if len(sub) < 3: continue
            naf = NelsonAalenFitter()
            naf.fit(sub["TBF_days"], sub["event"])
            h = float(naf.cumulative_hazard_.iloc[-1, 0])
            mtbf = sub["TBF_days"].mean()
            sev_pairs.append({DEV_KEY: dev, FC_COL: fc, "hazard": h, "mtbf": mtbf,
                              "severity": h + (1 - h / (h + 1))})

    if sev_pairs:
        sev_pair_df = pd.DataFrame(sev_pairs)
        pivot = sev_pair_df.pivot_table(index=DEV_KEY, columns=FC_COL, values="severity", aggfunc="mean")
        if _id_map is not None:
            pivot.index = pivot.index.map(lambda d: _id_map.get(d, d))

        fig, ax = plt.subplots(figsize=(14, 7))
        fig.suptitle(f"{DEVICE_TYPE} — Severity Heatmap: Device × Fault Code", fontsize=13, fontweight="bold")
        sns.heatmap(pivot, ax=ax, cmap="YlOrRd", linewidths=0.3, annot=True, fmt=".2f",
                    annot_kws={"size": 8}, cbar_kws={"label": "Severity Score"})
        ax.set(title="Severity by Device × Fault Code", xlabel="Fault Code",
               ylabel="Device ID" if _id_map is not None else "Device Key")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("[INFO] Not enough data for Device × Fault Code severity heatmap.")


In [ ]:
# ── Cell 21 ──────────────────────────────────────────────────────────
# ── Device Level Fault Occurrences for Top 20 Devices ─────────────────────────
if len(df) == 0 or not FC_COL:
    print("[SKIP]")
else:
    top20_devs = df[DEV_KEY].value_counts().nlargest(20).index.tolist()
    top10_fc_plot = df[FC_COL].value_counts().nlargest(10).index.tolist()

    pivot_occ = df[df[DEV_KEY].isin(top20_devs) & df[FC_COL].isin(top10_fc_plot)]\
        .groupby([DEV_KEY, FC_COL]).size().unstack(fill_value=0)
    if _id_map is not None:
        pivot_occ.index = pivot_occ.index.map(lambda d: _id_map.get(d, d))

    fig, ax = plt.subplots(figsize=(16, 9))
    fig.suptitle(f"{DEVICE_TYPE} — Fault Occurrence Heatmap: Top 20 Devices × Top 10 Fault Codes",
                 fontsize=13, fontweight="bold")

    sns.heatmap(pivot_occ, ax=ax, cmap="YlOrRd", linewidths=0.3,
                annot=True, fmt="d", annot_kws={"size": 7},
                cbar_kws={"label": "Occurrence Count"})
    ax.set(title="Fault Occurrences per Device", xlabel="Fault Code",
           ylabel="Device ID" if _id_map is not None else "Device Key")
    ax.tick_params(axis="x", rotation=45, labelsize=9)
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 22 ──────────────────────────────────────────────────────────
# ── Crow-AMSAA Reliability Growth Model ───────────────────────────────────────
if len(df) == 0:
    print("[SKIP]")
else:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(f"{DEVICE_TYPE} — Crow-AMSAA Reliability Growth + Bathtub Curve + MCF",
                 fontsize=13, fontweight="bold")

    # 1. Crow-AMSAA: cumulative failures per fault code over time
    top5_fc_crow = df[FC_COL].value_counts().nlargest(5).index.tolist() if FC_COL else []
    pal = sns.color_palette("tab10", 5)
    for i, fc in enumerate(top5_fc_crow):
        sub = df[df[FC_COL] == fc].sort_values(DATE_COL)
        monthly = sub.groupby(sub[DATE_COL].dt.to_period("M")).size().cumsum()
        axes[0].plot(range(len(monthly)), monthly.values, "o-", ms=4, lw=1.5,
                     label=f"FC={fc}", color=pal[i])
    axes[0].set(title="Crow-AMSAA: Cumulative Failures per Fault Code",
                xlabel="Month Index", ylabel="Cumulative Failures")
    axes[0].legend(fontsize=8)

    # 2. Bathtub curve: TBF density histogram
    if len(df_tbf) > 0:
        tbf_clip = df_tbf["TBF_days"].clip(upper=df_tbf["TBF_days"].quantile(0.98))
        axes[1].hist(tbf_clip, bins=60, density=True, color="#FF6F00", alpha=0.75, edgecolor="white")
        axes[1].set(title="Bathtub Curve (TBF Density)", xlabel="TBF (days)", ylabel="Density")

    # 3. MCF (Mean Cumulative Function) via Nelson-Aalen for top-5 fault codes
    if len(df_tbf) > 0 and FC_COL:
        for i, fc in enumerate(top5_fc_crow):
            sub = df_tbf[df_tbf[FC_COL] == fc]
            if len(sub) >= 5:
                naf = NelsonAalenFitter()
                naf.fit(sub["TBF_days"], sub["event"], label=f"FC={fc}")
                naf.plot_cumulative_hazard(ax=axes[2], ci_show=False, color=pal[i], lw=1.5)
    axes[2].set(title="MCF (Mean Cumulative Function) per Fault Code",
                xlabel="Days", ylabel="Mean Cumulative Failures")
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
# ── Cell 23 ──────────────────────────────────────────────────────────
# ── Reliability Block Diagram (RBD) — Separate diagram per device ────────────
if len(df) == 0 or not FC_COL:
    print("[SKIP]")
else:
    import math
    import matplotlib.cm as _cm
    import matplotlib.colors as _mc

    top5_rbd = df[DEV_KEY].value_counts().nlargest(5).index.tolist()

    for dev in top5_rbd:
        sub    = df[df[DEV_KEY] == dev]
        fc_cts = sub[FC_COL].value_counts().head(8)
        if fc_cts.empty:
            continue

        _did    = _id_map.get(dev, dev) if _id_map is not None else dev
        dev_lbl = f"Device {_did}"

        G = nx.DiGraph()
        G.add_node(dev_lbl, node_type="device")
        fc_nodes = []
        for fc, cnt in fc_cts.items():
            lbl = f"Fault Code {fc}"
            G.add_node(lbl, node_type="fault", count=int(cnt))
            G.add_edge(dev_lbl, lbl, weight=int(cnt))
            fc_nodes.append(lbl)

        # Fixed star layout — device at centre, FCs at equal angles starting from top
        n   = len(fc_nodes)
        pos = {dev_lbl: (0.0, 0.0)}
        for i, fn in enumerate(fc_nodes):
            ang     = 2 * math.pi * i / n - math.pi / 2
            pos[fn] = (math.cos(ang) * 2.2, math.sin(ang) * 2.2)

        # Node colors/sizes — red (high occurrence) → blue (low occurrence)
        max_cnt  = fc_cts.max()
        fc_norm  = [fc_cts.iloc[i] / max_cnt for i in range(n)]
        fc_colors = [_mc.to_hex(_cm.RdYlBu_r(v)) for v in fc_norm]
        fc_sizes  = [400 + 1600 * v for v in fc_norm]

        fig, ax = plt.subplots(figsize=(10, 8))
        ax.set_title(f"Reliability Block Diagram (RBD) for Device {_did}",
                     fontsize=12, fontweight="bold", pad=15)

        # Nodes
        nx.draw_networkx_nodes(G, pos, nodelist=[dev_lbl],
                               node_color=["#1565C0"], node_size=[2800],
                               ax=ax, alpha=0.92)
        nx.draw_networkx_nodes(G, pos, nodelist=fc_nodes,
                               node_color=fc_colors, node_size=fc_sizes,
                               ax=ax, alpha=0.85)

        # Edges — width scales with occurrence count
        ew = [G[u][v]["weight"] for u, v in G.edges()]
        nx.draw_networkx_edges(G, pos, ax=ax,
                               width=[3.0 * w / max_cnt for w in ew],
                               edge_color="#555555", arrows=True, arrowsize=18,
                               connectionstyle="arc3,rad=0.05",
                               min_source_margin=52, min_target_margin=28)

        # Labels — white on device node, dark on FC nodes
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_weight="bold",
                                font_color="white", labels={dev_lbl: dev_lbl})
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_color="#111111",
                                labels={fn: fn for fn in fc_nodes})

        ax.axis("off")
        plt.tight_layout()
        plt.show()

        # Per-device summary table
        sub_tbf = (df_tbf[df_tbf[DEV_KEY] == dev].groupby(FC_COL)["TBF_days"].mean().round(1)
                   if len(df_tbf) > 0 else pd.Series(dtype=float))
        print(f"\nDevice {_did} — Top Fault Codes ({len(sub):,} total incidents):")
        for fc, cnt in fc_cts.items():
            avg_t = sub_tbf.get(fc, float("nan"))
            print(f"  Fault Code {fc}: {cnt:,} occurrences  avg_TBF={avg_t:.1f}d")

    # ── Combined heatmap: top-5 devices × top-8 fault codes ──────────────────
    top5_fc_rbd = df[FC_COL].value_counts().nlargest(8).index.tolist()
    sub_rbd    = df[df[DEV_KEY].isin(top5_rbd) & df[FC_COL].isin(top5_fc_rbd)]
    pivot_rbd  = sub_rbd.groupby([DEV_KEY, FC_COL]).size().unstack(fill_value=0)
    if not pivot_rbd.empty:
        if _id_map is not None:
            pivot_rbd.index = pivot_rbd.index.map(lambda d: _id_map.get(d, d))
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.set_title(f"{DEVICE_TYPE} — Fault Occurrences: Top 5 Devices × Top 8 Fault Codes",
                     fontsize=11, fontweight="bold")
        sns.heatmap(pivot_rbd, ax=ax, cmap="Blues", annot=True, fmt="d",
                    annot_kws={"size": 9}, linewidths=0.3,
                    cbar_kws={"label": "Occurrence Count"})
        ax.set(xlabel="Fault Code",
               ylabel="Device ID" if _id_map is not None else "Device Key")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Cell 24 ──────────────────────────────────────────────────────────
# ── RUL Calculation by Different Methods ──────────────────────────────────────
# Method 1: Standard = avg_TBF × (n+1)
# Method 2: Decay Factor = avg_TBF × (n+1) × 0.85
# Method 3: Rolling Window (window=3)
if len(df_tbf) == 0:
    print("[SKIP]")
else:
    rul = df_tbf.sort_values([DEV_KEY, FC_COL, DATE_COL] if DATE_COL in df_tbf.columns else [DEV_KEY, FC_COL]).copy()
    rul["rolling_avg_tbf"] = rul.groupby([DEV_KEY, FC_COL])["TBF_days"].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean())

    # Per device
    rul_dev = df_tbf.groupby(DEV_KEY).agg(
        avg_tbf=("TBF_days", "mean"),
        total_failures=("TBF_days", "count"),
    ).reset_index()
    rul_dev["rolling_tbf"] = df_tbf.sort_values([DEV_KEY, DATE_COL]).groupby(DEV_KEY)["TBF_days"]\
        .transform(lambda x: x.rolling(3, min_periods=1).mean()).groupby(df_tbf[DEV_KEY]).last().reindex(rul_dev[DEV_KEY]).values

    n = rul_dev["total_failures"]
    rul_dev["RUL_standard"] = rul_dev["avg_tbf"] * (n + 1)
    rul_dev["RUL_decay"]    = rul_dev["avg_tbf"] * (n + 1) * 0.85
    rul_dev["RUL_rolling"]  = rul_dev["rolling_tbf"].fillna(rul_dev["avg_tbf"]) * np.minimum(n + 1, 5)

    # Per fault code
    rul_fc = df_tbf.groupby(FC_COL).agg(
        avg_tbf=("TBF_days", "mean"),
        total_failures=("TBF_days", "count"),
    ).reset_index()
    n_fc = rul_fc["total_failures"]
    rul_fc["RUL_standard"] = rul_fc["avg_tbf"] * (n_fc + 1)
    rul_fc["RUL_decay"]    = rul_fc["avg_tbf"] * (n_fc + 1) * 0.85
    rul_fc["RUL_rolling"]  = rul_fc["avg_tbf"] * np.minimum(n_fc + 1, 5)

    top30_dev = rul_dev.nlargest(min(30, len(rul_dev)), "total_failures")
    top10_fc_rul = rul_fc.nlargest(min(10, len(rul_fc)), "total_failures")

    print(f"{'='*70}")
    print(f"RUL Calculation — {DEVICE_TYPE} Devices (top 30 by failure count)")
    print(f"{'='*70}")
    _t23 = top30_dev.copy()
    if _id_map is not None:
        _t23.insert(0, DEV_ID_COL, _t23[DEV_KEY].map(_id_map))
    _c23 = ([DEV_ID_COL, DEV_KEY] if _id_map is not None else [DEV_KEY]) + ["total_failures", "avg_tbf", "RUL_standard", "RUL_decay", "RUL_rolling"]
    print(_t23[_c23].round(1).to_string(index=False))

    print(f"\n{'='*70}")
    print(f"RUL Calculation — {DEVICE_TYPE} Fault Codes (top 10 by frequency)")
    print(f"{'='*70}")
    print(top10_fc_rul[[FC_COL, "total_failures", "avg_tbf",
                         "RUL_standard", "RUL_decay", "RUL_rolling"]].round(1).to_string(index=False))

    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle(f"{DEVICE_TYPE} — RUL Calculation by Different Methods", fontsize=14, fontweight="bold")

    # Device RUL comparison
    x = np.arange(len(top30_dev)); w = 0.28
    axes[0,0].bar(x-w, top30_dev["RUL_standard"], w, label="Standard",   color="#1565C0", alpha=0.85)
    axes[0,0].bar(x,   top30_dev["RUL_decay"],    w, label="Decay ×0.85",color="#FF6F00", alpha=0.85)
    axes[0,0].bar(x+w, top30_dev["RUL_rolling"],  w, label="Rolling",    color="#2E7D32", alpha=0.85)
    axes[0,0].set_xticks(x)
    _xl23 = ([str(_id_map.get(d, d)) for d in top30_dev[DEV_KEY]]
             if _id_map is not None else [str(d) for d in top30_dev[DEV_KEY]])
    axes[0,0].set_xticklabels(_xl23, rotation=90, fontsize=6)
    axes[0,0].set(title="RUL per Device (top 30 by failure count)", ylabel="RUL (days)")
    axes[0,0].legend()

    # Fault code RUL comparison
    xf = np.arange(len(top10_fc_rul))
    axes[0,1].bar(xf-w, top10_fc_rul["RUL_standard"], w, label="Standard",  color="#1565C0", alpha=0.85)
    axes[0,1].bar(xf,   top10_fc_rul["RUL_decay"],    w, label="Decay×0.85",color="#FF6F00", alpha=0.85)
    axes[0,1].bar(xf+w, top10_fc_rul["RUL_rolling"],  w, label="Rolling",   color="#2E7D32", alpha=0.85)
    axes[0,1].set_xticks(xf)
    axes[0,1].set_xticklabels([f"FC={fc}" for fc in top10_fc_rul[FC_COL]], rotation=45, fontsize=8)
    axes[0,1].set(title="RUL per Fault Code (top 10 by frequency)", ylabel="RUL (days)")
    axes[0,1].legend()

    # Rolling window RUL — top 30 devices
    axes[1,0].bar(x, top30_dev["RUL_rolling"].values, color="#2E7D32", alpha=0.85)
    axes[1,0].set_xticks(x)
    axes[1,0].set_xticklabels(_xl23, rotation=90, fontsize=6)
    axes[1,0].set(title="Rolling Window RUL — Top 30 Devices", ylabel="RUL (days)")

    # Rolling window RUL — top 10 fault codes
    axes[1,1].bar(xf, top10_fc_rul["RUL_rolling"].values, color="#7B1FA2", alpha=0.85)
    axes[1,1].set_xticks(xf)
    axes[1,1].set_xticklabels([f"FC={fc}" for fc in top10_fc_rul[FC_COL]], rotation=45, fontsize=8)
    axes[1,1].set(title="Rolling Window RUL — Top 10 Fault Codes", ylabel="RUL (days)")

    plt.tight_layout()
    plt.show()

# ── MLflow: Device RUL Scoring ───────────────────────────────────────────────
try:
    with mlflow.start_run(run_name="device_rul_scoring") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type": "rul_estimation"})
        mlflow.log_params({
            "n_devices": len(top30_dev),
            "methods": "standard,decay,rolling",
        })
        for _, _row in top30_dev.iterrows():
            _rlbl = (str(_id_map.get(_row[DEV_KEY], _row[DEV_KEY]))
                     if _id_map is not None else str(_row[DEV_KEY]))
            mlflow.log_metric(f"rul_standard_{_rlbl}", float(_row["RUL_standard"]))
            mlflow.log_metric(f"rul_decay_{_rlbl}",    float(_row["RUL_decay"]))
            mlflow.log_metric(f"rul_rolling_{_rlbl}",  float(_row["RUL_rolling"]))
        _rcsv = os.path.join(OUT_DIR, "device_rul_estimates.csv")
        top30_dev.to_csv(_rcsv, index=False)
        mlflow.log_artifact(_rcsv, artifact_path="tables")
    print(f"[MLflow] RUL run: {_run.info.run_id}")
except Exception as _mle:
    print(f"[MLflow] RUL log skipped: {_mle}")

In [ ]:
# ── Feature Store : PS5 VALIDATOR RUL features ───────────────────────────────────
# Records per-device RUL survival estimates into SageMaker Feature Store.
# Offline store → S3 (batch inference / retraining)
# Online store  → DynamoDB (<10 ms, real-time device risk lookup)
# Set ENABLE_FEATURE_STORE = False to skip during dev reruns (saves ~5 min).

ENABLE_FEATURE_STORE = True

if not ENABLE_FEATURE_STORE:
    print("[SKIP] Feature Store disabled — set ENABLE_FEATURE_STORE=True to enable")
elif "rul_dev" not in dir() or len(rul_dev) == 0:
    print("[SKIP] No RUL data available — Feature Store ingestion skipped")
else:
    import time as _time, datetime as _dt

    # Build Feature Store dataframe from all-device RUL aggregation
    fs_df = rul_dev.copy()

    # Add DEVICE_ID column (mapped from DEVICE_KEY via gold PS1 table)
    if _id_map is not None:
        fs_df[DEV_ID_COL] = fs_df[DEV_KEY].map(_id_map).fillna(fs_df[DEV_KEY].astype(str))
    else:
        fs_df[DEV_ID_COL] = fs_df[DEV_KEY].astype(str)
    fs_df[DEV_ID_COL] = fs_df[DEV_ID_COL].astype(str)

    # Metadata columns
    fs_df["device_type"] = DEVICE_TYPE
    fs_df["event_time"]  = _dt.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

    # Select columns to ingest
    _fs_cols = [
        DEV_ID_COL, "avg_tbf", "total_failures", "rolling_tbf",
        "RUL_standard", "RUL_decay", "RUL_rolling",
        "device_type", "event_time",
    ]
    fs_df = fs_df[[c for c in _fs_cols if c in fs_df.columns]].copy()

    # Fill NAs so Feature Store doesn't reject records
    for _fc in fs_df.select_dtypes("float").columns:
        fs_df[_fc] = fs_df[_fc].fillna(0.0)

    print(f"Feature Store dataframe: {len(fs_df):,} rows  |  cols: {list(fs_df.columns)}")

    def _dtype_to_fs(dtype):
        if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    feature_group = FeatureGroup(
        name                = FS_GROUP_NAME,
        sagemaker_session   = SM_SESSION,
        feature_definitions = [
            FeatureDefinition(feature_name=c, feature_type=_dtype_to_fs(fs_df[c].dtype))
            for c in fs_df.columns
        ],
    )

    # Delete-and-recreate so schema changes land cleanly (PS3/PS4 pattern)
    _sm_fg_client = SM_SESSION.boto_session.client("sagemaker", region_name=REGION)
    _deleted = False
    try:
        _sm_fg_client.delete_feature_group(FeatureGroupName=FS_GROUP_NAME)
        print(f"Deleted existing Feature Group: {FS_GROUP_NAME}")
        _deleted = True
    except Exception:
        pass  # did not exist — fine
    if _deleted:
        print("Waiting 60s for deletion to propagate...")
        _time.sleep(60)

    try:
        feature_group.create(
            s3_uri                  = f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
            record_identifier_name  = DEV_ID_COL,
            event_time_feature_name = "event_time",
            role_arn                = ROLE,
            enable_online_store     = True,
            description             = (
                f"PS5 TVM device RUL survival features — Chicago Ventra CTA. "
                "Grain: device. Architecture: CUBIC MARS AWS v6."
            ),
        )
        _fg_status = ""
        while _fg_status != "Created":
            _time.sleep(15)
            _fg_status = feature_group.describe().get("FeatureGroupStatus")
            print(f"  Feature Group status: {_fg_status}")
        print(f"Feature Group created: {FS_GROUP_NAME}")
    except Exception as _fg_e:
        print(f"Feature Group create: {_fg_e}")

    feature_group.ingest(
        data_frame  = fs_df.reset_index(drop=True),
        max_workers = 4,
        wait        = True,
    )
    print(f"Ingested {len(fs_df):,} records → {FS_GROUP_NAME}")

    # Online store verify — spot-check first record
    _sample_id = fs_df[DEV_ID_COL].iloc[0]
    try:
        _record = feature_group.get_record(
            record_identifier_value_as_string=str(_sample_id))
        print(f"\nOnline store sample ({_sample_id}):")
        for _rf in _record[:4]:
            print(f"  {_rf['FeatureName']}: {_rf['ValueAsString']}")
    except Exception as _ve:
        print(f"Online store verify: {_ve}")

In [ ]:
# ── MLflow Model Registry : Register Cox PH as Champion ──────────────────────
mlfc = MlflowClient()
try:
    mlfc.create_registered_model(
        name=MLFLOW_MODEL_NAME,
        description=(
            f"PS5 {DEVICE_TYPE} Remaining Useful Life — Chicago Ventra CTA. "
            "Cox PH survival model. Grain: device x event. Architecture: CUBIC MARS AWS v6."
        ),
        tags={"ps": "PS5", "device_type": DEVICE_TYPE,
              "target": "rul_days", "city": "chicago"},
    )
    print(f"Created registered model: {MLFLOW_MODEL_NAME}")
except mlflow.exceptions.MlflowException:
    print(f"Already exists: {MLFLOW_MODEL_NAME}")

if CHAMPION_RUN_ID:
    mv = mlfc.create_model_version(
        name        = MLFLOW_MODEL_NAME,
        source      = f"runs:/{CHAMPION_RUN_ID}/model",
        run_id      = CHAMPION_RUN_ID,
        description = f"CoxPHFitter  CI={CHAMPION_CI:.4f}",
        tags={
            "algorithm":         "CoxPHFitter",
            "concordance_index": str(round(CHAMPION_CI, 4)),
            "device_type":       DEVICE_TYPE,
        },
    )
    mlfc.transition_model_version_stage(
        name=MLFLOW_MODEL_NAME, version=mv.version,
        stage="Staging", archive_existing_versions=False,
    )
    MLFLOW_VERSION = mv.version
    print(f"MLflow Model: {MLFLOW_MODEL_NAME}  v{MLFLOW_VERSION}  -> Staging")
    print(f"  To promote: mlfc.transition_model_version_stage(")
    print(f"      name='{MLFLOW_MODEL_NAME}', version='{MLFLOW_VERSION}',")
    print(f"      stage='Production', archive_existing_versions=True)")
else:
    print("[SKIP] No Champion Cox PH model — registry skipped")

In [ ]:
# ── FINAL CELL : Export notebook with all outputs to HTML ───────────────────
# Permanent fix for SageMaker Studio output stripping.
# Run this once at the end of a full notebook run.
# The HTML file is saved to EFS (/home/sagemaker-user/PS5/) and persists
# across sessions, instance stops, and kernel restarts.
# Open it in any browser to see all cell outputs, tables, and charts.

import subprocess, os, datetime

_nb_path  = "/home/sagemaker-user/PS5/PS5_TVM_Reliability_Analysis.ipynb"
_ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out_html = f"/home/sagemaker-user/PS5/PS5_TVM_Reliability_Analysis_output_{_ts}.html"

if os.path.exists(_nb_path):
    _result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", _nb_path, "--output", _out_html],
        capture_output=True, text=True
    )
    if _result.returncode == 0:
        _size = os.path.getsize(_out_html) / 1024
        print(f"[HTML EXPORT] Saved → {_out_html}  ({_size:.0f} KB)")
        print(f"  Open in browser — all cell outputs preserved permanently.")
    else:
        print(f"[HTML EXPORT] nbconvert error:")
        print(_result.stderr[-500:] if _result.stderr else "(no stderr)")
        _log = _out_html.replace(".html", ".log")
        print(f"  Fallback: write key results to {_log} manually.")
else:
    print(f"[HTML EXPORT] Notebook not found at {_nb_path}")
    print(f"  Actual path on EFS may differ — update _nb_path above.")
    print("  Run: !find /home/sagemaker-user -name 'PS5_TVM_Reliability_Analysis.ipynb' to locate it.")
